###  SISTEMA DE QUALIDADE DE DADOS — 6 PILARES
Pilares avaliados:
1. Completude    — valores ausentes / nulos
2. Consistência  — duplicatas, conflitos de tipo
3. Conformidade  — formatos esperados (datas, padrões)
4. Integridade   — relacionamentos e valores inválidos
5. Precisão      — outliers e valores suspeitos
6. Atualidade    — quão recentes são os registros

In [58]:
import sys
import os
import re
import warnings
from datetime import datetime, date
 
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from io import BytesIO
 
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, HRFlowable, Image
)
from reportlab.graphics.shapes import Drawing, Rect, String
from reportlab.graphics import renderPDF
 
warnings.filterwarnings("ignore")

In [59]:
#  Constantes e configurações globais
CONFIG = {
    # Colunas que NUNCA devem ser nulas (lista de nomes exatos)
    "colunas_obrigatorias": [],
 
    # Colunas que devem ser únicas (ex: CPF, ID)
    "colunas_unicas": [],
 
    # Formatos de data aceitos (o script testa todos automaticamente)
    "formatos_data": ["%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%Y/%m/%d", "%d.%m.%Y"],
 
    # Colunas que são datas (deixe vazio para detecção automática)
    "colunas_data": [],
 
    # Número de dias: registros mais antigos que isso geram alerta de atualidade
    "atualidade_limite_dias": 365,
 
    # Limiar de outlier via IQR (padrão 1.5)
    "outlier_iqr_fator": 1.5,
 
    # Porcentagem mínima aceitável de completude por coluna (0–100)
    "completude_minima_pct": 80.0,
}
 
 
# Entrada e saída
ARQUIVO_CSV = r"../dataset/amazon_prime_titles.csv" 
ARQUIVO_PDF = None

#  Paleta de cores
COR_PRIMARIA   = colors.HexColor("#1B3A6B")   # azul escuro
COR_SECUNDARIA = colors.HexColor("#2E86C1")   # azul médio
COR_DESTAQUE   = colors.HexColor("#F39C12")   # âmbar
COR_SUCESSO    = colors.HexColor("#27AE60")   # verde
COR_ALERTA     = colors.HexColor("#E74C3C")   # vermelho
COR_NEUTRO     = colors.HexColor("#ECF0F1")   # cinza claro
COR_TEXTO      = colors.HexColor("#2C3E50")   # quase preto

## 1. LEITURA E PRÉ-PROCESSAMENTO

In [60]:
def carregar_csv(caminho: str) -> pd.DataFrame:
    """Tenta ler o CSV com diferentes encodings e separadores."""
    encodings = ["utf-8", "latin-1", "cp1252", "utf-8-sig"]
    separadores = [",", ";", "\t", "|"]
    for enc in encodings:
        for sep in separadores:
            try:
                df = pd.read_csv(caminho, encoding=enc, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    return df
            except Exception:
                continue
    raise ValueError(f"Não foi possível ler o arquivo: {caminho}")
 
 
def detectar_colunas_data(df: pd.DataFrame) -> list:
    """Detecta automaticamente colunas que parecem conter datas."""
    candidatas = []
    palavras_chave = ["data", "date", "dt_", "_dt", "vencimento", "nascimento",
                      "criacao", "criação", "abertura", "fechamento", "prazo"]
    for col in df.columns:
        col_lower = col.lower()
        if any(k in col_lower for k in palavras_chave):
            candidatas.append(col)
            continue
        # Testa amostra da coluna
        amostra = df[col].dropna().astype(str).head(20)
        for fmt in CONFIG["formatos_data"]:
            try:
                pd.to_datetime(amostra, format=fmt, errors="raise")
                candidatas.append(col)
                break
            except Exception:
                continue
    return list(set(candidatas))
 
 
def tentar_parse_data(serie: pd.Series) -> pd.Series:
    """Tenta converter uma série para datetime usando os formatos configurados."""
    for fmt in CONFIG["formatos_data"]:
        try:
            return pd.to_datetime(serie, format=fmt, errors="coerce")
        except Exception:
            continue
    return pd.to_datetime(serie, infer_datetime_format=True, errors="coerce")

## 2. AVALIAÇÃO DOS 6 PILARES

### SEÇÃO: COMPLETUDE

In [61]:
def avaliar_completude(df: pd.DataFrame) -> dict:
    total_celulas = df.shape[0] * df.shape[1]
    nulos = df.isnull().sum()
    pct_nulo = (nulos / len(df) * 100).round(2)
    pct_preenchido = (100 - pct_nulo).round(2)
 
    # Colunas que ficam abaixo do mínimo configurado
    problemas = pct_preenchido[pct_preenchido < CONFIG["completude_minima_pct"]].to_dict()
 
    # Score: média ponderada de preenchimento
    score = float(pct_preenchido.mean())
 
    detalhe = pd.DataFrame({
        "Coluna": nulos.index,
        "Nulos": nulos.values,
        "% Preenchido": pct_preenchido.values
    }).sort_values("% Preenchido")
 
    # Alertas para colunas obrigatórias
    alertas_obrig = []
    for col in CONFIG["colunas_obrigatorias"]:
        if col in df.columns and df[col].isnull().any():
            n = int(df[col].isnull().sum())
            alertas_obrig.append(f"'{col}': {n} nulo(s) em coluna obrigatória")
 
    return {
        "score": round(score, 2),
        "total_nulos": int(nulos.sum()),
        "total_celulas": total_celulas,
        "detalhe": detalhe,
        "colunas_criticas": problemas,
        "alertas_obrigatorias": alertas_obrig,
    }

### SEÇÃO: CONSISTÊNCIA

In [62]:
def avaliar_consistencia(df: pd.DataFrame) -> dict:
    # Duplicatas completas
    dup_completas = int(df.duplicated().sum())
    pct_dup = round(dup_completas / len(df) * 100, 2)
 
    # Duplicatas por colunas únicas configuradas
    dup_unicas = {}
    for col in CONFIG["colunas_unicas"]:
        if col in df.columns:
            n = int(df[col].dropna().duplicated().sum())
            if n > 0:
                dup_unicas[col] = n
 
    # Inconsistências de tipo: colunas numéricas com strings misturadas
    tipo_misto = {}
    for col in df.select_dtypes(include="object").columns:
        amostra = df[col].dropna().head(500)
        n_num = amostra.apply(lambda x: str(x).replace(",", ".").replace("-", "")
                               .replace(" ", "").isnumeric()).sum()
        if 0 < n_num < len(amostra) * 0.9 and n_num > len(amostra) * 0.1:
            tipo_misto[col] = {"numerico": int(n_num), "texto": int(len(amostra) - n_num)}
 
    penalidade = (pct_dup / 100) * 50 + min(len(dup_unicas) * 10, 30) + min(len(tipo_misto) * 5, 20)
    score = max(0, 100 - penalidade)
 
    return {
        "score": round(score, 2),
        "duplicatas_completas": dup_completas,
        "pct_duplicatas": pct_dup,
        "duplicatas_colunas_unicas": dup_unicas,
        "tipo_misto": tipo_misto,
    }

### SEÇÃO: CONFORMIDADE

In [63]:
def avaliar_conformidade(df: pd.DataFrame, colunas_data: list) -> dict:
    problemas_data = {}
    datas_invalidas_total = 0
 
    for col in colunas_data:
        if col not in df.columns:
            continue
        serie = df[col].dropna().astype(str)
        parsed = tentar_parse_data(serie)
        invalidas = int(parsed.isnull().sum())
        if invalidas > 0:
            problemas_data[col] = invalidas
            datas_invalidas_total += invalidas
 
    total_datas = sum(df[c].notna().sum() for c in colunas_data if c in df.columns)
    pct_invalido = (datas_invalidas_total / total_datas * 100) if total_datas > 0 else 0
    score = max(0, 100 - pct_invalido)
 
    return {
        "score": round(score, 2),
        "colunas_data_analisadas": colunas_data,
        "datas_invalidas": problemas_data,
        "total_datas_invalidas": datas_invalidas_total,
        "total_datas": int(total_datas),
    }

### SEÇÃO: INTEGRIDADE

In [64]:
def avaliar_integridade(df: pd.DataFrame) -> dict:
    problemas = {}
 
    # Verifica valores negativos em colunas que deveriam ser positivas
    colunas_num = df.select_dtypes(include=[np.number]).columns
    palavras_positivas = ["valor", "preco", "preço", "quantidade", "qtd", "qtde",
                          "total", "age", "idade", "salario", "salário"]
    negativos = {}
    for col in colunas_num:
        if any(p in col.lower() for p in palavras_positivas):
            n_neg = int((df[col] < 0).sum())
            if n_neg > 0:
                negativos[col] = n_neg
 
    # Verifica strings vazias (whitespace only)
    strings_vazias = {}
    for col in df.select_dtypes(include="object").columns:
        n = int((df[col].astype(str).str.strip() == "").sum())
        if n > 0:
            strings_vazias[col] = n
 
    total_problemas = sum(negativos.values()) + sum(strings_vazias.values())
    total_registros = len(df)
    penalidade = min((total_problemas / total_registros) * 100, 100) if total_registros > 0 else 0
    score = max(0, 100 - penalidade)
 
    return {
        "score": round(score, 2),
        "valores_negativos_suspeitos": negativos,
        "strings_vazias": strings_vazias,
        "total_problemas": total_problemas,
    }


### SEÇÃO: PRECISÃO

In [65]:
def avaliar_precisao(df: pd.DataFrame) -> dict:
    outliers = {}
    fator = CONFIG["outlier_iqr_fator"]
    colunas_num = df.select_dtypes(include=[np.number]).columns
 
    for col in colunas_num:
        serie = df[col].dropna()
        if len(serie) < 10:
            continue
        Q1 = serie.quantile(0.25)
        Q3 = serie.quantile(0.75)
        IQR = Q3 - Q1
        if IQR == 0:
            continue
        limite_inf = Q1 - fator * IQR
        limite_sup = Q3 + fator * IQR
        n_out = int(((serie < limite_inf) | (serie > limite_sup)).sum())
        if n_out > 0:
            outliers[col] = {
                "outliers": n_out,
                "pct": round(n_out / len(serie) * 100, 2),
                "limite_inf": round(float(limite_inf), 4),
                "limite_sup": round(float(limite_sup), 4),
                "min": round(float(serie.min()), 4),
                "max": round(float(serie.max()), 4),
            }
 
    total_out = sum(v["outliers"] for v in outliers.values())
    total_vals = sum(df[c].notna().sum() for c in colunas_num)
    pct_out = (total_out / total_vals * 100) if total_vals > 0 else 0
    score = max(0, 100 - pct_out * 2)
 
    return {
        "score": round(score, 2),
        "outliers_por_coluna": outliers,
        "total_outliers": total_out,
        "pct_outliers": round(pct_out, 2),
    }

### SEÇÃO: ATUALIDADE

In [66]:
def avaliar_atualidade(df: pd.DataFrame, colunas_data: list) -> dict:
    hoje = pd.Timestamp.today()
    limite_dias = CONFIG["atualidade_limite_dias"]
    analise = {}
 
    for col in colunas_data:
        if col not in df.columns:
            continue
        parsed = tentar_parse_data(df[col].dropna().astype(str))
        parsed = parsed.dropna()
        if len(parsed) == 0:
            continue
        diff = (hoje - parsed).dt.days
        desatualizados = int((diff > limite_dias).sum())
        media_dias = float(diff.mean())
        mais_recente = parsed.max()
        mais_antigo = parsed.min()
        analise[col] = {
            "registros_analisados": len(parsed),
            "desatualizados": desatualizados,
            "pct_desatualizados": round(desatualizados / len(parsed) * 100, 2),
            "media_dias_atraso": round(media_dias, 1),
            "data_mais_recente": str(mais_recente.date()),
            "data_mais_antiga": str(mais_antigo.date()),
        }
 
    if analise:
        media_pct = np.mean([v["pct_desatualizados"] for v in analise.values()])
        score = max(0, 100 - media_pct)
    else:
        score = 100.0
 
    return {
        "score": round(score, 2),
        "analise_por_coluna": analise,
        "limite_dias": limite_dias,
    }

## 3. GERAÇÃO DE GRÁFICOS

In [67]:
def grafico_radar(scores: dict) -> BytesIO:
    """Gráfico radar com os 6 pilares."""
    labels = list(scores.keys())
    valores = list(scores.values())
    N = len(labels)
 
    angulos = [n / float(N) * 2 * np.pi for n in range(N)]
    angulos += angulos[:1]
    valores_plot = valores + valores[:1]
 
    fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor("white")
 
    ax.set_facecolor("#F8F9FA")
    ax.plot(angulos, valores_plot, "o-", linewidth=2, color="#2E86C1")
    ax.fill(angulos, valores_plot, alpha=0.25, color="#2E86C1")
 
    ax.set_xticks(angulos[:-1])
    ax.set_xticklabels(labels, size=9, fontweight="bold", color="#2C3E50")
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(["20", "40", "60", "80", "100"], size=7, color="#7F8C8D")
    ax.grid(color="#BDC3C7", linestyle="--", linewidth=0.5)
    ax.spines["polar"].set_color("#BDC3C7")
 
    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    buf.seek(0)
    return buf
 
 
def grafico_barras_completude(detalhe: pd.DataFrame) -> BytesIO:
    """Barras horizontais com % de preenchimento por coluna (top 15 piores)."""
    piores = detalhe.head(15)
    fig, ax = plt.subplots(figsize=(7, max(3, len(piores) * 0.45)))
    fig.patch.set_facecolor("white")
 
    cores = ["#E74C3C" if v < CONFIG["completude_minima_pct"] else "#27AE60"
             for v in piores["% Preenchido"]]
    bars = ax.barh(piores["Coluna"], piores["% Preenchido"], color=cores, height=0.6)
 
    for bar, val in zip(bars, piores["% Preenchido"]):
        ax.text(min(val + 1, 98), bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", ha="left", fontsize=8, color="#2C3E50")
 
    ax.axvline(x=CONFIG["completude_minima_pct"], color="#F39C12",
               linestyle="--", linewidth=1.5, label=f"Mínimo ({CONFIG['completude_minima_pct']}%)")
    ax.set_xlim(0, 105)
    ax.set_xlabel("% Preenchido", fontsize=9)
    ax.set_title("Completude por Coluna (piores)", fontsize=10, fontweight="bold", color="#1B3A6B")
    ax.legend(fontsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.set_facecolor("#F8F9FA")
    plt.tight_layout()
 
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    buf.seek(0)
    return buf
 
 
def grafico_score_geral(score_geral: float) -> BytesIO:
    """Gauge simples para o score geral."""
    fig, ax = plt.subplots(figsize=(4, 2.5))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 5)
    ax.axis("off")
 
    # Fundo do gauge
    for i, (inicio, fim, cor) in enumerate([
        (0, 3.33, "#E74C3C"), (3.33, 6.66, "#F39C12"), (6.66, 10, "#27AE60")
    ]):
        ax.barh(2, fim - inicio, left=inicio, height=1.2,
                color=cor, alpha=0.3, edgecolor="white", linewidth=2)
 
    # Marcador
    pos = score_geral / 10
    ax.annotate("", xy=(pos * 10, 2.6), xytext=(pos * 10, 1.5),
                arrowprops=dict(arrowstyle="->", color="#1B3A6B", lw=2.5))
 
    ax.text(5, 4.2, f"Score Geral: {score_geral:.1f} / 100",
            ha="center", va="center", fontsize=13, fontweight="bold", color="#1B3A6B")
    ax.text(1.5, 0.5, "Crítico\n(0–33)", ha="center", fontsize=7.5, color="#E74C3C")
    ax.text(5, 0.5, "Moderado\n(34–66)", ha="center", fontsize=7.5, color="#F39C12")
    ax.text(8.5, 0.5, "Bom\n(67–100)", ha="center", fontsize=7.5, color="#27AE60")
 
    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    buf.seek(0)
    return buf

## 4. GERAÇÃO DO PDF

### Prepação

In [68]:
def grafico_radar(scores: dict) -> BytesIO:
    """Gráfico radar com os 6 pilares."""
    labels = list(scores.keys())
    valores = list(scores.values())
    N = len(labels)
 
    angulos = [n / float(N) * 2 * np.pi for n in range(N)]
    angulos += angulos[:1]
    valores_plot = valores + valores[:1]
 
    fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor("white")
 
    ax.set_facecolor("#F8F9FA")
    ax.plot(angulos, valores_plot, "o-", linewidth=2, color="#2E86C1")
    ax.fill(angulos, valores_plot, alpha=0.25, color="#2E86C1")
 
    ax.set_xticks(angulos[:-1])
    ax.set_xticklabels(labels, size=9, fontweight="bold", color="#2C3E50")
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(["20", "40", "60", "80", "100"], size=7, color="#7F8C8D")
    ax.grid(color="#BDC3C7", linestyle="--", linewidth=0.5)
    ax.spines["polar"].set_color("#BDC3C7")
 
    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    buf.seek(0)
    return buf
 
 
def grafico_barras_completude(detalhe: pd.DataFrame) -> BytesIO:
    """Barras horizontais com % de preenchimento por coluna (top 15 piores)."""
    piores = detalhe.head(15)
    fig, ax = plt.subplots(figsize=(7, max(3, len(piores) * 0.45)))
    fig.patch.set_facecolor("white")
 
    cores = ["#E74C3C" if v < CONFIG["completude_minima_pct"] else "#27AE60"
             for v in piores["% Preenchido"]]
    bars = ax.barh(piores["Coluna"], piores["% Preenchido"], color=cores, height=0.6)
 
    for bar, val in zip(bars, piores["% Preenchido"]):
        ax.text(min(val + 1, 98), bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", ha="left", fontsize=8, color="#2C3E50")
 
    ax.axvline(x=CONFIG["completude_minima_pct"], color="#F39C12",
               linestyle="--", linewidth=1.5, label=f"Mínimo ({CONFIG['completude_minima_pct']}%)")
    ax.set_xlim(0, 105)
    ax.set_xlabel("% Preenchido", fontsize=9)
    ax.set_title("Completude por Coluna (piores)", fontsize=10, fontweight="bold", color="#1B3A6B")
    ax.legend(fontsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.set_facecolor("#F8F9FA")
    plt.tight_layout()
 
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    buf.seek(0)
    return buf
 
 
def grafico_score_geral(score_geral: float) -> BytesIO:
    """Gauge simples para o score geral."""
    fig, ax = plt.subplots(figsize=(4, 2.5))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 5)
    ax.axis("off")
 
    # Fundo do gauge
    for i, (inicio, fim, cor) in enumerate([
        (0, 3.33, "#E74C3C"), (3.33, 6.66, "#F39C12"), (6.66, 10, "#27AE60")
    ]):
        ax.barh(2, fim - inicio, left=inicio, height=1.2,
                color=cor, alpha=0.3, edgecolor="white", linewidth=2)
 
    # Marcador
    pos = score_geral / 10
    ax.annotate("", xy=(pos * 10, 2.6), xytext=(pos * 10, 1.5),
                arrowprops=dict(arrowstyle="->", color="#1B3A6B", lw=2.5))
 
    ax.text(5, 4.2, f"Score Geral: {score_geral:.1f} / 100",
            ha="center", va="center", fontsize=13, fontweight="bold", color="#1B3A6B")
    ax.text(1.5, 0.5, "Crítico\n(0–33)", ha="center", fontsize=7.5, color="#E74C3C")
    ax.text(5, 0.5, "Moderado\n(34–66)", ha="center", fontsize=7.5, color="#F39C12")
    ax.text(8.5, 0.5, "Bom\n(67–100)", ha="center", fontsize=7.5, color="#27AE60")
 
    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    buf.seek(0)
    return buf

### Geração

In [69]:
# ...existing code...

def gerar_pdf(caminho_csv: str, df: pd.DataFrame, resultados: dict, scores: dict, score_geral: float, colunas_data: list, caminho_saida: str):
    """Gera o relatório PDF com os resultados da análise."""
    doc = SimpleDocTemplate(caminho_saida, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []
    
    # Título
    title = Paragraph("Relatório de Qualidade de Dados", styles['Title'])
    story.append(title)
    story.append(Spacer(1, 12))
    
    # Resumo geral
    resumo = f"Arquivo: {os.path.basename(caminho_csv)}<br/>Registros: {len(df):,}<br/>Colunas: {df.shape[1]}<br/>Score Geral: {score_geral:.1f}/100"
    story.append(Paragraph(resumo, styles['Normal']))
    story.append(Spacer(1, 12))
    
    # Gráfico radar
    buf_radar = grafico_radar(scores)
    img_radar = Image(buf_radar, width=300, height=300)
    story.append(img_radar)
    story.append(Spacer(1, 12))
    
    # Gráfico score geral
    buf_gauge = grafico_score_geral(score_geral)
    img_gauge = Image(buf_gauge, width=200, height=125)
    story.append(img_gauge)
    story.append(Spacer(1, 12))
    
    # Detalhes por pilar
    for pilar, res in resultados.items():
        story.append(Paragraph(f"<b>{pilar.capitalize()}</b> - Score: {res['score']:.1f}/100", styles['Heading2']))
        if pilar == "completude":
            buf_barras = grafico_barras_completude(res["detalhe"])
            img_barras = Image(buf_barras, width=400, height=200)
            story.append(img_barras)
        # Adicione mais detalhes conforme necessário
        story.append(Spacer(1, 12))
    
    # Construir PDF
    doc.build(story)
    print(f"   ✔ Relatório salvo em: {caminho_saida}")

## 5. EXECUÇÃO PRINCIPAL

In [70]:
def executar(caminho_csv: str, caminho_saida: str = None):
    print(f"\n📂 Carregando arquivo: {caminho_csv}")
    df = carregar_csv(caminho_csv)
    print(f"   ✔ {len(df):,} registros | {df.shape[1]} colunas")
 
    # Detecta colunas de data
    colunas_data = CONFIG["colunas_data"] or detectar_colunas_data(df)
    if colunas_data:
        print(f"   📅 Colunas de data detectadas: {colunas_data}")
 
    print("\n🔍 Avaliando os 6 pilares...")
    resultados = {
        "completude":   avaliar_completude(df),
        "consistencia": avaliar_consistencia(df),
        "conformidade": avaliar_conformidade(df, colunas_data),
        "integridade":  avaliar_integridade(df),
        "precisao":     avaliar_precisao(df),
        "atualidade":   avaliar_atualidade(df, colunas_data),
    }
 
    scores = {
        "Completude":  resultados["completude"]["score"],
        "Consistência": resultados["consistencia"]["score"],
        "Conformidade": resultados["conformidade"]["score"],
        "Integridade": resultados["integridade"]["score"],
        "Precisão":    resultados["precisao"]["score"],
        "Atualidade":  resultados["atualidade"]["score"],
    }
 
    score_geral = round(sum(scores.values()) / len(scores), 2)
 
    for pilar, sc in scores.items():
        status = "✅" if sc >= 80 else ("⚠️" if sc >= 60 else "❌")
        print(f"   {status} {pilar:<15} {sc:.1f}/100")
    print(f"\n   📊 SCORE GERAL: {score_geral:.1f}/100")
 
    if not caminho_saida:
        base = os.path.splitext(os.path.basename(caminho_csv))[0]
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        caminho_saida = os.path.join(os.path.dirname(caminho_csv), f"relatorio_qualidade_{base}_{ts}.pdf")
 
    print("\n📄 Gerando relatório PDF...")
    gerar_pdf(caminho_csv, df, resultados, scores, score_geral, colunas_data, caminho_saida)  # type: ignore[name-defined]
    return caminho_saida
 
 
# Detecta ambiente de execução
EM_JUPYTER = "ipykernel" in sys.modules
 
if not EM_JUPYTER:
    # Fora do Jupyter, aceita argumentos via linha de comando:
    #   python data_quality.py arquivo.csv [saida.pdf]
    if len(sys.argv) >= 2:
        ARQUIVO_CSV = sys.argv[1]
    if len(sys.argv) >= 3:
        ARQUIVO_PDF = sys.argv[2]
 
if not os.path.isfile(ARQUIVO_CSV):
    print(f"\n❌ Arquivo não encontrado: {ARQUIVO_CSV}")
    print("   Verifique o caminho na variável ARQUIVO_CSV no topo do bloco.")
else:
    executar(ARQUIVO_CSV, ARQUIVO_PDF)


📂 Carregando arquivo: ../dataset/amazon_prime_titles.csv
   ✔ 9,668 registros | 12 colunas
   📅 Colunas de data detectadas: ['date_added']

🔍 Avaliando os 6 pilares...
   ✅ Completude      80.9/100
   ✅ Consistência    100.0/100
   ❌ Conformidade    0.0/100
   ✅ Integridade     100.0/100
   ⚠️ Precisão        77.8/100
   ✅ Atualidade      100.0/100

   📊 SCORE GERAL: 76.5/100

📄 Gerando relatório PDF...
   ✔ Relatório salvo em: ../dataset/relatorio_qualidade_amazon_prime_titles_20260314_190923.pdf
